[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajeevraibhatia/agent-harness-evals/blob/main/building-agents/notebooks/m1_first_agent.ipynb) [![Course](https://img.shields.io/badge/Course-rajeevraibhatia.com-7c3aed)](https://rajeevraibhatia.com/curriculum/building-agents#module-1)

# Module 1 — Your First Agent

**Level:** Beginner | **Time:** ~30 min | [Full reading →](https://rajeevraibhatia.com/curriculum/building-agents#module-1)

### What you'll build
A working agent with one tool in 25 lines. No frameworks.

### Key concepts
- **Chatbot vs agent**: tool schema + loop = agent
- The model *requests* tools; your code *runs* them
- Loop exits when no tool calls are returned

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
# Option A: OpenAI API (recommended for Colab)
!pip install openai --quiet

import os
from openai import OpenAI

# Set your OpenAI API key — in Colab: Secrets (🔑) → add OPENAI_API_KEY
# Then enable notebook access, or paste directly (don't commit keys to git)
# os.environ["OPENAI_API_KEY"] = "sk-..."   # ← uncomment and paste if not using Secrets

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
MODEL = "gpt-4o"

# ── Option B: Ollama (local, no API key, no cost) ─────────────────────────────
# 1. Install Ollama: https://ollama.com/download
# 2. Run: ollama pull llama3.2
# 3. Uncomment the two lines below and comment out the OpenAI lines above:
#
# client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# MODEL = "llama3.2"   # or: mistral, phi4, gemma3, qwen2.5, etc.
#
# Everything in this notebook works with either client — MODEL is passed through.
print(f"Client ready. Using model: {MODEL}")

In [ ]:
import json
import math
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

TOOLS = [{
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Evaluate a math expression. Example: '847 * 0.15'",
        "parameters": {
            "type": "object",
            "properties": {"expression": {"type": "string"}},
            "required": ["expression"]
        }
    }
}]

def run_tool(name: str, args: dict) -> str:
    if name == "calculator":
        try:
            return str(eval(args["expression"], {"__builtins__": {}}, vars(math)))
        except Exception as e:
            return f"Error: {e}"
    return f"Unknown tool: {name}"

def run_agent(question: str, max_steps: int = 10) -> str:
    messages = [{"role": "user", "content": question}]
    for step in range(max_steps):
        r = client.chat.completions.create(model="gpt-4o", messages=messages, tools=TOOLS)
        msg = r.choices[0].message
        messages.append(msg)
        if not msg.tool_calls:
            return msg.content
        for tc in msg.tool_calls:
            result = run_tool(tc.function.name, json.loads(tc.function.arguments))
            print(f"  [{step}] {tc.function.name}({tc.function.arguments}) → {result}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    return "Max steps reached."

print(run_agent("What is 15% of 847?"))

## Exercise

Add a `string_length(text)` tool that returns the number of characters in a string.

In [ ]:
# Exercise: add a string_length tool
# 1. Write the JSON schema and add it to TOOLS
# 2. Add the handler in run_tool()
# 3. Test: "How many characters are in 'supercalifragilistic'?"
# 4. Bonus: ask a question that uses BOTH tools

# Your code here: